# 04 Clustering: Lifestyle Profiles


## Objective

修正聚类输入范围：聚类训练只使用数字行为与生活习惯数值特征；人口背景、设备类别和结果变量只在聚类后用于画像解释。


In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd().resolve().parent if (Path.cwd().resolve().parent / "src").exists() else PROJECT_ROOT

sys.path.insert(0, str(PROJECT_ROOT / "src"))
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

from config import FIGURES_DIR, RANDOM_STATE, RESULTS_DIR
from data_utils import ensure_project_dirs, load_processed_dataset
from feature_engineering import make_feature_target
from model_utils import run_clustering_experiment
from visualization import (
    plot_cluster_pca,
    plot_cluster_profile_heatmap,
    plot_clustering_model_comparison,
    plot_clustering_scores,
    plot_kmeans_elbow,
    plot_silhouette_by_k,
)

np.random.seed(RANDOM_STATE)
ensure_project_dirs()


PROJECT_ROOT = C:\Users\qintian\Desktop\大数据\Big-Data-Homework\期末考查报告_数字生活方式分析


## Clustering Feature Whitelist

下面的特征矩阵应只包含行为与生活习惯数值列，不包含 `gender`、`region`、`daily_role`、`device_type` 等背景/类别变量，也不包含任何 outcome 字段。


In [2]:
df = load_processed_dataset(fallback_to_raw=True)
X_cluster, _ = make_feature_target(df, task="clustering")
print(f"Clustering feature matrix shape: {X_cluster.shape}")
display(pd.DataFrame({"clustering_feature": X_cluster.columns}))


Clustering feature matrix shape: (3500, 15)


,clustering_feature
0,device_hours_per_day
1,phone_unlocks
2,notifications_per_day
3,social_media_mins
4,study_mins
5,physical_activity_days
6,sleep_hours
7,sleep_quality
8,social_media_hours
9,study_hours


## Clustering Algorithm Comparison

比较 KMeans、AgglomerativeClustering 和 GaussianMixture，k 从 2 到 8；KMeans 额外输出 inertia 用于肘部法则。


In [3]:
clustering_result = run_clustering_experiment(df)
print(f"Selected model: {clustering_result['best_algorithm']}, k={clustering_result['best_k']}")
display(clustering_result["model_comparison"])
display(clustering_result["profile"])


Selected model: kmeans, k=3


,algorithm,k,inertia,silhouette,calinski_harabasz,davies_bouldin
0,agglomerative,2,NaN,0.134642,590.105158,2.073101
1,agglomerative,3,NaN,0.151769,495.893794,1.876770
2,agglomerative,4,NaN,0.149575,429.188941,1.812652
3,agglomerative,5,NaN,0.100314,392.239672,2.063345
4,agglomerative,6,NaN,0.079029,373.868510,2.111110
5,agglomerative,7,NaN,0.087878,363.527529,1.912063
6,agglomerative,8,NaN,0.080312,348.794901,1.889007
7,gaussian_mixture,2,NaN,0.147831,650.846092,2.159854
8,gaussian_mixture,3,NaN,0.093362,467.291544,2.770252
9,gaussian_mixture,4,NaN,0.122486,408.172480,2.226574


,cluster,cluster_algorithm,cluster_size,device_hours_per_day,phone_unlocks,notifications_per_day,social_media_mins,study_mins,physical_activity_days,sleep_hours,sleep_quality,social_media_hours,study_hours,notifications_per_device_hour,unlocks_per_device_hour,device_to_sleep_ratio,activity_sleep_interaction,social_to_study_ratio,high_risk_flag,productivity_score,digital_dependence_score,anxiety_score,depression_score,stress_level,happiness_score,focus_score,gender_mode,region_mode,income_level_mode,education_level_mode,daily_role_mode,device_type_mode,suggested_cluster_label
0,0,kmeans,447,6.953311,140.950783,296.225951,430.212528,95.067114,3.398210,7.381558,2.789936,7.170209,1.584452,48.323716,20.630966,0.984150,25.039615,84.138701,0.196868,65.337548,34.329610,6.010113,8.458613,4.779337,6.713225,25.561947,Female,Europe,Low,Bachelor,Full-time Employee,Laptop,high_social_media_profile
1,1,kmeans,1039,11.053782,220.039461,337.102984,131.878730,109.794033,3.090472,6.121790,1.695291,2.197979,1.829901,31.809441,20.247838,1.893336,18.779143,13.027026,0.372474,65.908133,51.242518,11.286477,14.534167,6.489515,5.051457,42.583226,Female,Europe,Lower-Mid,Bachelor,Full-time Employee,Tablet,high_device_dependence_profile
2,2,kmeans,2014,5.471132,110.849057,342.724429,113.427507,110.147468,3.473188,7.810619,3.213733,1.890458,1.835791,72.591520,20.903095,0.720152,27.136491,13.633520,0.114201,64.976723,29.696236,5.327872,5.969712,4.413600,7.058447,44.649919,Female,Europe,Low,Bachelor,Full-time Employee,Android,balanced_low_load_profile


## Clustering Figures

若轮廓系数仍然偏低，后续报告只能把聚类写成探索性用户画像，而不是严格的人群边界。


In [4]:
plot_kmeans_elbow(clustering_result["scores"], FIGURES_DIR / "clustering_kmeans_elbow.png")
plot_silhouette_by_k(clustering_result["model_comparison"], FIGURES_DIR / "clustering_silhouette_by_k.png")
plot_clustering_model_comparison(clustering_result["model_comparison"], FIGURES_DIR / "clustering_model_comparison.png")
plot_clustering_scores(clustering_result["scores"], FIGURES_DIR / "clustering_kmeans_selection.png")
plot_cluster_pca(clustering_result["pca_coordinates"], FIGURES_DIR / "clustering_lifestyle_pca.png")
plot_cluster_profile_heatmap(clustering_result["profile"], FIGURES_DIR / "clustering_lifestyle_profile_heatmap.png")
print("Saved clustering figures.")


Saved clustering figures.
